# Betsson live-odds scraper

Workflow:
1. Open a Playwright session and pre-open one browser tab per sport in `SPORTS`.
2. Call `await session.get_odds(sport=...)` for any sport — that tab is reused, the events container is scrolled top→bottom→top while scraping at each step, results are deduped by `event_id`, and a parquet snapshot is written to `data/snapshots/betsson_<sport>_<UTC>.parquet`.
3. `await session.close()` when done.

If the page redirects or renders blank, you're likely geo-blocked — connect via a Lithuanian VPN.

In [ ]:
import sys
from pathlib import Path
import random
import time

sys.path.insert(0, str(Path.cwd().parent))

from scrapers.betsson.page import open_page

SPORTS = ["basketball", "football"]
HEADLESS = False  # flip to True if you don't have WSLg / a display server

In [ ]:
# Open the session and pre-open one tab per sport
session = await open_page(headless=HEADLESS)
for sport in SPORTS:
    await session.open_sport(sport)
print(f"open tabs: {list(session.tabs)}")

In [ ]:
# Scrape every configured sport in turn (each call writes its own snapshot)
dfs = {}
for sport in SPORTS:
    df = await session.get_odds(sport=sport)
    dfs[sport] = df
    print(f"{sport}: {df.shape[0]} events scraped")

In [ ]:
# Re-scrape a single sport on demand (re-run for a fresh snapshot)
df = await session.get_odds(sport="basketball")
print(df.shape)
df.head(20)

In [ ]:
for i in range(2):
    for sport in SPORTS:
        df = await session.get_odds(sport=sport)
    sleep_duration = random.uniform(5, 12)
    time.sleep(sleep_duration)

In [ ]:
# List the parquet snapshots that have been written so far
snap_dir = Path.cwd().parent / "data" / "snapshots"
for p in sorted(snap_dir.glob("betsson_*.parquet")):
    print(p.relative_to(Path.cwd().parent), p.stat().st_size, "bytes")

In [ ]:
await session.close()